# Clustering

### **CAKE** (**C**onfidence in **A**ssignments via **K**-partition **E**nsembles)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics import silhouette_samples, confusion_matrix
from scipy.optimize import linear_sum_assignment
from sklearn.preprocessing import LabelEncoder

def sil_samples(X, labels, approximation=False, centers=None):
    """
    Compute silhouette scores for each point in the dataset,
    with approximate fast centroid-based computation option.
    """
    # Ensure arrays
    X = np.asarray(X)
    labels = np.asarray(labels)
    if X.ndim != 2:
       raise ValueError("X must be a 2D array of shape (n_samples, n_features).")
    if labels.ndim != 1 or labels.shape[0] != X.shape[0]:
       raise ValueError("labels must be a 1D array of length n_samples.")

    unique_labels, inv = np.unique(labels, return_inverse=True)
    k = unique_labels.size
    if k < 2:
       raise ValueError("Silhouette computation requires at least 2 clusters.")

    # Exact silhouette scores
    if approximation == False:
       silhouette_scores = silhouette_samples(X, labels=labels)
       return silhouette_scores

    # Centroid-based approximate silhouette scores
    n_samples, n_features = X.shape

    if centers is None:
       centers = np.array([X[inv == i].mean(axis=0) for i in range(k)], dtype=float)
    else:
       centers = np.asarray(centers, dtype=float)
       if centers.ndim != 2 or centers.shape[1] != n_features:
          raise ValueError(f"centers must have shape (k, d) with d={n_features}.")
       if centers.shape[0] != k:
          raise ValueError(f"centers.shape[0] must equal number of clusters k={k}.")
       if not np.array_equal(unique_labels, np.arange(k)):
          raise ValueError("When passing ndarray centers, labels must be dense 0..k-1.")

    # Squared distances to all centroids
    D_sq = euclidean_distances(X, centers, squared=True)

    # a(i): distance to own centroid
    a = np.sqrt(np.maximum(D_sq[np.arange(n_samples), inv], 0.0))

    # b(i): distance to nearest other centroid
    D_sq[np.arange(n_samples), inv] = np.inf
    b = np.sqrt(np.min(D_sq, axis=1))

    # Silhouette per point
    denom = np.maximum(np.maximum(a, b), 1e-12)
    s_point = (b - a) / denom

    # Singleton clusters -> silhouette = 0
    counts = np.bincount(inv, minlength=k).astype(int)
    s_point[counts[inv] < 2] = 0.0

    silhouette_scores = np.clip(s_point, -1.0, 1.0)

    return silhouette_scores

def sil_samples_stats(X, labels_list, approximation=False, centers_list=None):
    """
    Compute and aggregate silhouette scores across multiple clustering runs,
    returning per-sample mean and standard deviation of silhouette scores.
    """
    X = np.asarray(X)
    n_samples = X.shape[0]
    n_runs = len(labels_list)
    if n_runs < 2:
       raise ValueError("Clustering Ensemble must contain at least 2 partitions.")

    if centers_list is not None and len(centers_list) != n_runs:
       raise ValueError("centers_list must match the number of labelings in labels_list")

    # Initialize matrix to hold silhouette scores
    sil_scores = np.zeros((n_runs, n_samples), dtype=float)

    for i, labels in enumerate(labels_list):
        labels_arr = np.asarray(labels)
        centers = None
        if approximation and centers_list is not None:
           centers = centers_list[i]

        # Compute silhouette scores for this run
        sil_scores[i] = sil_samples(X, labels_arr, approximation=approximation, centers=centers)

    # Compute mean and standard deviation of sample-silhouette scores over the ensemble
    mean_sil_samples = sil_scores.mean(axis=0)
    std_sil_samples = sil_scores.std(axis=0)

    return mean_sil_samples, std_sil_samples

def align_labels(target, source):
    """
    Aligns the labels in "source" to match the labels in "target"
    using the Hungarian Algorithm based on a contingency matrix.

    Parameters:
    - target: array-like of shape (n_samples,)
              The reference label vector to align to.
    - source: array-like of shape (n_samples,)
              The label vector to be permuted for alignment.

    Returns:
    - aligned: np.ndarray of shape (n_samples,)
               The source labels, remapped to best match the target labels.
    """
    target = np.asarray(target)
    source = np.asarray(source)
    unique_target = np.unique(target)
    unique_source = np.unique(source)
    if len(unique_target) != len(unique_source):
       raise ValueError(
           f"Cannot align: Target has {len(unique_target)} clusters {unique_target.tolist()}, "
           f"but source has {len(unique_source)} clusters {unique_source.tolist()}"
       )
    # Encode labels to indices based on their unique values
    le_target = LabelEncoder().fit(unique_target)
    le_source = LabelEncoder().fit(unique_source)

    target_encoded = le_target.transform(target)
    source_encoded = le_source.transform(source)

    # Confusion matrix with indices aligned to encoded labels
    c_matrix = confusion_matrix(target_encoded, source_encoded)

    # Apply the Hungarian algorithm
    row_ind, col_ind = linear_sum_assignment(-c_matrix)

    # Create mapping from source labels to target labels
    mapping = {
        le_source.classes_[src_col]: le_target.classes_[tgt_row]
        for tgt_row, src_col in zip(row_ind, col_ind)
    }
    # Remap source labels using the mapping
    aligned = np.vectorize(mapping.get)(source)

    return aligned

def pairwise_stability(labels_runs):
    """
    Computes per-point clustering stability across multiple runs by aligning labels
    pairwise using the Hungarian algorithm to account for label permutations.

    Parameters:
    - labels_runs: list of array-like
        List of label arrays from multiple clustering runs. Each array must have the
        same number of samples and clusters (unique labels).

    Returns:
    - stability: np.ndarray of shape (n_samples,)
        Per-point stability score between 0 and 1, indicating the fraction of run-pairs
        where the point's label matches after optimal alignment.

    Notes:
    - Requires all runs to have the same number of clusters (unique labels).
    - Label alignment is done pairwise between runs using the Hungarian algorithm.
    - High stability (~1) indicates stable cluster assignments across runs.
    """
    labels_runs = [np.asarray(labels) for labels in labels_runs]
    n_runs, n_samples = len(labels_runs), len(labels_runs[0])
    if n_runs < 2:
       raise ValueError("pairwise_stability needs at least 2 runs.")

    # Counters: How many run-pairs agree per point
    agreement_counts = np.zeros(n_samples, dtype=int)
    total_pairs = 0

    # For every unique pair of runs
    for r1 in range(n_runs):
        labels_r1 = labels_runs[r1]
        for r2 in range(r1+1, n_runs):
            labels_r2 = labels_runs[r2]

            # Align labels_r2 to labels_r1 using Hungarian
            aligned_r2 = align_labels(labels_r1, labels_r2)
            matches = (labels_r1 == aligned_r2)

            # Add matches to total per-point agreement count
            agreement_counts += matches
            total_pairs += 1

    return agreement_counts/total_pairs

def cake(X, labels_list, method='product', approximation=False, centers_list=None, geom_norm='clip'):
    """
    Compute a confidence score per point for clustering ensembles, defined as:
    stability_i * geometric_stability_i or
    2 * stability_i * geometric_stability_i / [stability_i + geometric_stability_i]
    where stability is the pairwise label agreement across runs,
    and mean_sil_i, std_sil_i are the statistics from sil_samples_stats.

    Parameters:
    - X: array-like, shape (n_samples, n_features)
    - labels_list: list of array-like, each of shape (n_samples,)
        A list of cluster labelings for the same dataset X.
    - approximation: bool, default=False
        Whether to use the approximate silhouette computation.
    - centers_list: list of pd.Series or array-like, optional
        If approximation=True, an optional list of centroids for each clustering.
    - method: str, default='product'
        'product' or 'harmonic_mean' for the formulation of the CAKE scores.
    - geo_norm: str, default='clip'
        - 'affine' (default): maps geom_raw in [-1,1] to [0,1] via (x+1)/2 (preserves negatives).
        - 'clip': max(mean_sil - std_sil, 0) clipped at 1 (original behavior; discards negatives).

    Returns:
    - cake_scores: np.ndarray, shape (n_samples,)
        The CAKE scores for each point.
    - stability: np.ndarray, shape (n_samples,)
        The stability scores for each point.
    - geom_stability: np.ndarray, shape (n_samples,)
        The silhouette-based reliability scores for each point (mean_sil_i - std_sil_i).
    - summary: pd.DataFrame
        Summary of above metrics per point in X.
    """
    # Compute silhouette statistics
    mean_sil, std_sil = sil_samples_stats(X, labels_list,
                                          approximation=approximation,
                                          centers_list=centers_list)

    # Assignment stability across the ensemble
    stability = pairwise_stability(labels_list)

    # Silhouette-based stability across the ensemble
    geom_raw = mean_sil - std_sil
    if geom_norm == 'clip':
       geom_stability = np.clip(mean_sil - std_sil, 0.0, 1.0)
    else:
       geom_stability = np.clip((geom_raw + 1) / 2, 0, 1) # preserves information for negative scores

    # Calculate confidence scores
    if method == 'product':
       cake_scores = stability * geom_stability
    elif method == 'harmonic_mean':
       num = 2.0 * stability * geom_stability
       denom = np.maximum(stability + geom_stability, 1e-8)
       cake_scores = num / denom
    else:
       raise ValueError(
           f"Unknown method: {method}. Use 'product' or 'harmonic_mean'."
       )

    # Summary DataFrame
    summary = pd.DataFrame({
        'Mean Silhouette': mean_sil,
        'STD Silhouette': std_sil,
        'Geometric Stability': geom_stability,
        'Assignment Stability': stability,
        'CAKE': cake_scores
    })

    return cake_scores, stability, geom_stability, summary

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from collections import OrderedDict
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, Normalize

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    silhouette_samples,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from scipy.stats import spearmanr
from scipy.stats import t
from sklearn.metrics import f1_score, adjusted_mutual_info_score
from scipy.stats import kendalltau, spearmanr
from sklearn.mixture import GaussianMixture
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.decomposition import PCA
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
import time, gc

def clustering_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    accuracy = cm[row_ind, col_ind].sum() / np.sum(cm)
    return accuracy

def micro_sil(X, labels, approximation=False, centers=None):
    s = sil_samples(X, labels, approximation=approximation, centers=centers)
    return float(np.mean(s))

def macro_sil(X, labels, approximation=False, centers=None):
    labels = np.asarray(labels)
    s = sil_samples(X, labels, approximation=approximation, centers=centers)
    uniq = np.unique(labels)
    cluster_means = [s[labels == lab].mean() for lab in uniq]
    return float(np.mean(cluster_means))

def consensus_medoid_majority(labels_runs):
    if not labels_runs:
        raise ValueError("labels_runs must be a non-empty list of 1D arrays.")
    labs = [np.asarray(l) for l in labels_runs]
    R = len(labs)
    n = labs[0].shape[0]
    if any(len(l) != n for l in labs):
        raise ValueError("All runs must have the same number of samples.")

    ks = {np.unique(l).size for l in labs}
    if len(ks) != 1:
        raise ValueError(f"All runs must have the same number of clusters; got {sorted(ks)}.")

    # pairwise symmetric Hamming distances after alignment
    D = np.zeros((R, R), dtype=float)
    for i in range(R):
        li = labs[i]
        for j in range(i+1, R):
            lj = labs[j]
            aligned_j = align_labels(li, lj)
            d_ij = 1.0 - np.mean(aligned_j == li)
            aligned_i = align_labels(lj, li)
            d_ji = 1.0 - np.mean(aligned_i == lj)
            D[i, j] = D[j, i] = 0.5 * (d_ij + d_ji)

    medoid = int(np.argmin(D.sum(axis=1)))
    ref = labs[medoid]
    aligned = [align_labels(ref, l) for l in labs]   # align all runs to the medoid
    A = np.vstack(aligned)                           # (R, n)

    # majority vote per point; tie -> medoid label
    z_star = np.empty(n, dtype=ref.dtype)
    for i in range(n):
        counts = Counter(A[:, i])
        top = counts.most_common(2)
        if len(top) == 1 or top[0][1] > top[1][1]:
            z_star[i] = top[0][0]
        else:
            z_star[i] = ref[i]  # tie-break to medoid label

    agree = np.mean(A == z_star, axis=0)
    return z_star, agree

### Data: Normalization, PCA, and $k$ Selection

**Embeddings**

In [ ]:
import numpy as np

file_path = './out_files/par_embeddings.npy'

try:
    X = np.load(file_path)
    print("loaded")
    print(f"\nEmbeddings (Samples, Dimensions): {X.shape}")
except FileNotFoundError:
    print(f"File not found: {file_path}")

**Paragraphs**

In [ ]:
import pandas as pd

file_path = './out_files/01_paragraphs_clean.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Shape: {df.shape}")
    display(df.head(3))
except FileNotFoundError:
    print(f"File not found: {file_path}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Scale the data (just like before)
X_scaled = StandardScaler().fit_transform(X)

# 2. Fit PCA with exactly 100 components
pca_100 = PCA(n_components=100, random_state=42)
pca_100.fit(X_scaled) # We only need to fit it to check the variance

# 3. Calculate the cumulative explained variance
explained_100 = pca_100.explained_variance_ratio_.cumsum()

# 4. Grab the final value (which represents the total variance for all 100 components)
total_variance_100 = explained_100[-1]

print(f"Total variance explained by 100 components: {total_variance_100:.4f} ({total_variance_100 * 100:.2f}%)")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=100, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"New shape: {X_pca.shape}")

### Micro- and Macro-averaged Silhouette Scores

In [ ]:
import os

os.makedirs('./out_files/plots/cake/', exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt

kmin, kmax, step = 15, 101, 1

k_range = range(kmin, kmax, step)
micro_scores = []
macro_scores = []

print("Starting K-selection tests (with centroid approximation)...")

for k_test in k_range:
    km_test = KMeans(n_clusters=k_test, random_state=42, n_init=3)
    labels_test = km_test.fit_predict(X_pca)
    centroids_test = km_test.cluster_centers_

    mi_s = micro_sil(X_pca, labels_test, approximation=True, centers=centroids_test)
    ma_s = macro_sil(X_pca, labels_test, approximation=True, centers=centroids_test)

    micro_scores.append(mi_s)
    macro_scores.append(ma_s)

    print(f"k={k_test} | Micro-Sil: {mi_s:.4f} | Macro-Sil: {ma_s:.4f}")

plt.figure(figsize=(12, 6))
plt.plot(k_range, micro_scores, 'o-', label='Micro Silhouette (Standard)', color='blue')
plt.plot(k_range, macro_scores, 's--', label='Macro Silhouette (Per Cluster)', color='red')

plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.savefig('./out_files/plots/cake/01_K-Selection_Analysis.jpg', dpi=500, bbox_inches='tight')
plt.show()

In [ ]:
micro_np = np.array(micro_scores)
macro_np = np.array(macro_scores)

harmonic_mean_scores = 2 * (micro_np * macro_np) / (micro_np + macro_np + 1e-9)

idx_micro = np.argmax(micro_np)
idx_macro = np.argmax(macro_np)
idx_harmonic = np.argmax(harmonic_mean_scores)

best_k_micro = k_range[idx_micro]
best_k_macro = k_range[idx_macro]
best_k_harmonic = k_range[idx_harmonic]

print(f"--- Final K-Selection Results ---\n")
print(f"Best K (Micro-Sil):    {best_k_micro}  (Score: {micro_np[idx_micro]:.4f})")
print(f"Best K (Macro-Sil):    {best_k_macro}  (Score: {macro_np[idx_macro]:.4f})")
print(f"Best K (Harmonic):     {best_k_harmonic}  (Score: {harmonic_mean_scores[idx_harmonic]:.4f})")

plt.figure(figsize=(12, 6))
plt.plot(k_range, harmonic_mean_scores, 'g-', label='Harmonic Mean (Combined)', linewidth=2.5)

plt.plot(k_range, micro_np, 'b--', alpha=0.3, label='Micro-Silhouette')
plt.plot(k_range, macro_np, 'r--', alpha=0.3, label='Macro-Silhouette')

plt.axvline(x=best_k_harmonic, color='darkred', linestyle=':',
            label=f'Optimal K (Harmonic) = {best_k_harmonic}')

plt.xlabel('Number of Clusters (k)')
plt.ylabel('Score Value')
plt.legend()
plt.grid(True, alpha=0.2)
plt.savefig('./out_files/plots/cake/02_K-Selection_Strategy.jpg', dpi=500, bbox_inches='tight')
plt.show()

### Clustering Ensemble and CAKE Scores

In [ ]:
from sklearn.cluster import KMeans

k_final = best_k_harmonic
n_runs = 10
labels_list = []
centers_list = []

print(f"Running CAKE Ensemble for k={k_final}...")

for i in range(n_runs):

    km = KMeans(n_clusters=k_final, random_state=i, n_init=3)
    labels = km.fit_predict(X_pca)

    labels_list.append(labels)
    centers_list.append(km.cluster_centers_)

    if (i+1) % 2 == 0:
        print(f"Completed run {i+1}/{n_runs}")

print("Ensemble Construction: Finished")

cake_scores, stability, geom_stability, summary = cake(
    X_pca,
    labels_list,
    method='harmonic_mean',
    approximation=True,
    centers_list=centers_list,
    geom_norm='affine'
)

print("\nCAKE Scores Computed.\n")

**Consensus clustering and CAKE summary**

In [ ]:
z_star, agreement = consensus_medoid_majority(labels_list)
summary['Final_Cluster'] = z_star
summary['Agreement_Rate'] = agreement

display(summary.head(5))

### CAKE scores distribution

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

sns.histplot(summary['CAKE'], bins=50, kde=True, color='teal', edgecolor='black', alpha=0.7)

plt.axvline(summary['CAKE'].mean(), color='red', linestyle='--', label=f"Mean: {summary['CAKE'].mean():.2f}")
plt.axvline(summary['CAKE'].median(), color='orange', linestyle='-', label=f"Median: {summary['CAKE'].median():.2f}")

plt.xlabel('CAKE Score (Reliability)', fontsize=12)
plt.ylabel('Frequency (Number of Paragraphs)', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.savefig('./out_files/plots/cake/03_CAKE_Score_Distribution.jpg', dpi=500, bbox_inches='tight')
plt.show()

### Consensus Clustering Analysis

In [ ]:
# Consensus clustering aggrement stats
mean_agreement = summary['Agreement_Rate'].mean()
median_agreement = summary['Agreement_Rate'].median()
std_agreement = summary['Agreement_Rate'].std()

# Docs wirh Agreement == 1.0
unanimous_ratio = (summary['Agreement_Rate'] == 1.0).mean() * 100

print(f"\n--- Consensus Ensemble Statistics (n_runs={n_runs}) ---")
print(f"Mean Agreement: {mean_agreement:.4f}")
print(f"Median Agreement: {median_agreement:.4f}")
print(f"Standard Deviation: {std_agreement:.4f}")
print(f"Unanimous Decisions: {unanimous_ratio:.2f}% of total speeches")

# Cluster sizes stats
cluster_counts = summary['Final_Cluster'].value_counts().sort_index()
print("\n--- Cluster Size Distribution ---")
display(cluster_counts.describe())

**Silhouette Scores on Consensus Partition**

In [ ]:
# Calculate silhouette scores for the final consensus labels (z_star)
#  approximation=True for speed given the 60k samples
consensus_sil_scores = sil_samples(X_pca, z_star, approximation=True)

# Add scores to summary df
summary['Consensus_Silhouette'] = consensus_sil_scores

display(summary.head(2))

In [ ]:
# aggregate metrics per cluster using cluster labels
cluster_profiles = summary.groupby('Final_Cluster').agg(
    Size=('CAKE', 'count'),
    mean_CAKE=('CAKE', 'mean'),
    std_CAKE=('CAKE', 'std'),
    mean_Agreement=('Agreement_Rate', 'mean'),
    mean_Silhouette=('Consensus_Silhouette', 'mean') # Silhouette per cluster
).reset_index()

# percentage share of each cluster
cluster_profiles['share'] = (cluster_profiles['Size'] / len(summary)) * 100

# rename
cluster_profiles = cluster_profiles.rename(columns={'Final_Cluster': 'cluster_label'})

# cluster label as idx
final_report = cluster_profiles.set_index('cluster_label')

# reorder columns
final_report = final_report[['Size', 'share', 'mean_CAKE', 'std_CAKE', 'mean_Agreement', 'mean_Silhouette']]

# Calculate the Global Silhouette of the consensus partition
total_consensus_sil = consensus_sil_scores.mean()

print(f"Total Consensus Silhouette Score: {total_consensus_sil:.4f}")
print("\n--- Cluster Statistical Profiles (Consensus Partition) ---")

display(final_report)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(16, 6))

ax = sns.barplot(x=final_report.index, y='share', data=final_report, palette='magma', edgecolor='black')

for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=8, fontweight='bold', xytext=(0, 5),
                textcoords='offset points')


plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Percentage of Total Speeches (%)', fontsize=12)
plt.xticks(final_report.index)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('./out_files/plots/cake/04_Cluster_Share_Distribution.jpg', dpi=500, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(16, 6))
colors = sns.color_palette("coolwarm", len(final_report))
ax = sns.barplot(x=final_report.index, y='mean_Silhouette', data=final_report, palette=colors, edgecolor='black')

plt.axhline(y=final_report['mean_Silhouette'].mean(), color='red', linestyle='--', label='Global Mean')
# plt.title('Consensus Partition: Geometric Cohesion (Mean Silhouette)', fontsize=15)
plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.savefig('./out_files/plots/cake/05_Cluster_mean_Silhouette_Scores.jpg', dpi=500, bbox_inches='tight')
plt.show()

In [ ]:
display(summary)

### Filtering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

cutoff_30 = np.percentile(summary['CAKE'], 30)
stable_indices = np.where(summary['CAKE'] >= cutoff_30)[0]
X_stable = X_pca[stable_indices]
summary_stable = summary.iloc[stable_indices].copy()

print(f"--- Step 1: Denoising (CAKE >= 30th Percentile) ---")
print(f"Cutoff Value: {cutoff_30:.4f}")
print(f"Samples remaining: {len(X_stable)} ({len(X_stable)/len(X_pca)*100:.1f}% of original)")

k_range = range(15, 61, 1)
micro_scores = []
macro_scores = []

print("\n--- Step 2: Re-evaluating K for 70% Stable Subset ---")
for k_test in k_range:
    km_test = KMeans(n_clusters=k_test, random_state=42, n_init=3)
    labels_test = km_test.fit_predict(X_stable)
    mi_s = sil_samples(X_stable, labels_test, approximation=True, centers=km_test.cluster_centers_)
    micro_scores.append(mi_s.mean())
    temp_df = pd.DataFrame({'labels': labels_test, 'sil': mi_s})
    macro_scores.append(temp_df.groupby('labels')['sil'].mean().mean())

harmonic_mean_scores = 2 * (np.array(micro_scores) * np.array(macro_scores)) / (np.array(micro_scores) + np.array(macro_scores) + 1e-9)
best_k_stable = k_range[np.argmax(harmonic_mean_scores)]

km_final_stable = KMeans(n_clusters=best_k_stable, random_state=42, n_init=10)
z_star_stable = km_final_stable.fit_predict(X_stable)
stable_sil_scores = sil_samples(X_stable, z_star_stable, approximation=True, centers=km_final_stable.cluster_centers_)

summary_stable['Final_Cluster'] = z_star_stable
summary_stable['Consensus_Silhouette'] = stable_sil_scores

cluster_profiles = summary_stable.groupby('Final_Cluster').agg(
    Size=('CAKE', 'count'),
    mean_CAKE=('CAKE', 'mean'),
    std_CAKE=('CAKE', 'std'),
    mean_Agreement=('Agreement_Rate', 'mean'),
    mean_Silhouette=('Consensus_Silhouette', 'mean')
).reset_index()

cluster_profiles['share'] = (cluster_profiles['Size'] / len(summary_stable)) * 100
final_report_stable = cluster_profiles.rename(columns={'Final_Cluster': 'cluster_label'}).set_index('cluster_label')
final_report_stable = final_report_stable[['Size', 'share', 'mean_CAKE', 'std_CAKE', 'mean_Agreement', 'mean_Silhouette']]

display(final_report_stable)

plt.figure(figsize=(16, 6))
ax1 = sns.barplot(x=final_report_stable.index, y='share', data=final_report_stable, palette='magma', edgecolor='black')

for p in ax1.patches:
    ax1.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=8, fontweight='bold', xytext=(0, 5),
                textcoords='offset points')

plt.title(f'Stable Subset (K={best_k_stable}): Topic Share (%)', fontsize=15, pad=20)
plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('./out_files/plots/cake/06_filtered_Subset_Cluster_Share.jpg', dpi=500, bbox_inches='tight')
plt.show()

plt.figure(figsize=(16, 6))
colors = sns.color_palette("coolwarm", len(final_report_stable))
ax2 = sns.barplot(x=final_report_stable.index, y='mean_Silhouette', data=final_report_stable, palette=colors, edgecolor='black')

for p in ax2.patches:
    ax2.annotate(f'{p.get_height():.3f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=8, fontweight='bold', xytext=(0, 5),
                textcoords='offset points')

plt.axhline(y=final_report_stable['mean_Silhouette'].mean(), color='red', linestyle='--',
            label=f"Global Mean: {final_report_stable['mean_Silhouette'].mean():.4f}")
# plt.title(f'Stable Subset (K={best_k_stable}): Geometric Cohesion (Mean Silhouette)', fontsize=15)
plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./out_files/plots/cake/06_filtered_Subset_Cluster_mean_Silhouette.jpg', dpi=500, bbox_inches='tight')
plt.show()

### Save Clustered df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Filter the original dataset using the indices of the 70% we kept
df_stable = df.iloc[stable_indices].copy()

# Add the new cluster labels and the CAKE scores
df_stable['cluster_label'] = summary_stable['Final_Cluster'].values
df_stable['cake'] = summary_stable['CAKE'].values

# Save this high-confidence dataset for future use
output_stable_csv = './out_files/02_dataset_clustered.csv'
df_stable.to_csv(output_stable_csv, index=False, encoding='utf-8')
print(f"Stable clustered dataset saved to: {output_stable_csv}")

In [ ]:
df_stable.info()

### Clusters Metadata

In [ ]:
# Load stopwords from your text file
with open ('./out_files/01_stopwords_all.txt', 'r', encoding='utf-8') as f:
    stopwords_all = [line.strip() for line in f if line.strip()]

In [ ]:
# Group texts by cluster and concatenate them
# .astype(str) ensures we don't hit errors if there are empty or null rows
cluster_texts = (
    df_stable
    .groupby("cluster_label")["par_clean"]
    .apply(lambda texts: " ".join(texts.astype(str)))
)

# Initialize and fit the TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    ngram_range=(1,2), 
    max_df=0.9,
    min_df=1,
    max_features=10000,
    stop_words=stopwords_all, 
)

# Transform the concatenated cluster texts into a TF-IDF matrix
X_cluster_tfidf = vectorizer.fit_transform(cluster_texts)

# Extract the vocabulary (feature names)
terms = np.array(vectorizer.get_feature_names_out())

In [ ]:
top_n = 20
metadata_records = []

# Get the total number of documents per cluster
doc_counts = df_stable['cluster_label'].value_counts().to_dict()

# Pre-sort the dataframe to easily grab the top 3 speeches per cluster by CAKE score
top_par = (
    df_stable.sort_values(by=['cluster_label', 'cake'], ascending=[True, False])
    .groupby('cluster_label')
    .head(5)
)


In [ ]:
# Loop through each cluster to build its metadata
for i, cluster_id in enumerate(cluster_texts.index):
    
    # 1. Extract Top TF-IDF Terms for this cluster
    row = X_cluster_tfidf[i].toarray().flatten()
    top_idx = row.argsort()[-top_n:][::-1] # Get indices of the top N terms
    
    # Format the terms and their scores into a single readable string
    top_terms_str = ", ".join(
        [f"{terms[idx]} ({row[idx]:.3f})" for idx in top_idx if row[idx] > 0]
    )
    
    # 2. Extract Top 3 Paragraphs and CAKE scores for this cluster
    cluster_top_docs = top_par[top_par['cluster_label'] == cluster_id]
    
    par_list = []
    cake_scores_list = []
    
    # Loop through the top 3 docs 
    for rank, (_, doc_row) in enumerate(cluster_top_docs.iterrows(), 1):
        par_text = str(doc_row['paragraph'])
        
        # Truncate if over 2500 characters
        if len(par_text) > 2500:
            par_text = par_text[:2500] + "..."
            
        par_list.append(par_text)
        cake_scores_list.append(f"{doc_row['cake']:.3f}")
        
    # Join the lists together with " | " 
    top_paragraphs_str = " | ".join(par_list)
    top_cakes_str = " | ".join(cake_scores_list)
    
    # 3. Append all this information as a dictionary to our records list
    metadata_records.append({
        'Cluster_ID': cluster_id,
        'Cluster_title': "",
        'Doc_Count': doc_counts.get(cluster_id, 0),
        'Top_TFIDF_Terms': top_terms_str,
        'Top_Paragraphs': top_paragraphs_str,
        'Top_CAKE_Scores': top_cakes_str
    })

In [ ]:
# Convert the records into a DataFrame
cluster_metadata_df = pd.DataFrame(metadata_records)

display(cluster_metadata_df)


In [ ]:
# Save the metadata table to CSV
cluster_metadata_df.to_csv('./out_files/02_cluster_metadata.csv', index=False, encoding='utf-8')
print("Metadata saved successfully to: ./out_files/02_cluster_metadata.csv")

### LLM titles

In [ ]:
%pip install boto3

In [ ]:
# enter creedencials

import os
import boto3

def parse_env_credentials(filepath):
    """
    Parses AWS credentials from a text file with KEY=VALUE format.
    
    Args:
        filepath: Path to the credentials file (e.g., 'credentials.txt' or '.env').
        
    Returns:
        dict: A dictionary ready to be unpacked into boto3.client/resource,
              or None if parsing fails.
    """
    creds = {}
    
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                # Skip empty lines or comments
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                if '=' in line:
                    key, value = line.split('=', 1)
                    key = key.strip()
                    value = value.strip()
                    
                    # Map the file keys to boto3 expected arguments
                    if key == 'AWS_REGION':
                        creds['aws_region'] = value
                    elif key == 'AWS_ACCESS_KEY_ID':
                        creds['aws_access_key_id'] = value
                    elif key == 'AWS_SECRET_ACCESS_KEY':
                        creds['aws_secret_access_key'] = value
                    elif key == 'AWS_SESSION_TOKEN': 
                        # Added this just in case your new file uses temporary keys
                        creds['aws_session_token'] = value

        # specific check to ensure we found what we needed
        if 'aws_access_key_id' not in creds:
             print(f"Warning: No AWS_ACCESS_KEY_ID found in {filepath}")
             return None

        return creds

    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found.")
        return None
    except Exception as e:
        print(f"Error parsing file: {e}")
        return None



# 1. Point to your local file (e.g., "credentials.json")
file_path = "./credentials.json"

# 2. Parse
credentials = parse_env_credentials(file_path)

if credentials:
    print("Credentials parsed successfully!")
    print(f"Region: {credentials.get('aws_region')}")
    
    # 3. Use with Boto3 (Unpacking the dict automatically)
    
    
    bedrock_client = boto3.client(
        service_name='bedrock-runtime',
        aws_access_key_id=credentials['aws_access_key_id'],
        aws_secret_access_key=credentials['aws_secret_access_key'],
        # IMPORTANT: boto3 expects the argument 'region_name', so we map 'aws_region' to it
        region_name=credentials['aws_region']
    )
    
    # 3. Test (Optional print to verify)
    print(f"Client created for region: {credentials['aws_region']}")
    # The **credentials syntax automatically expands the dict into arguments:
    # aws_access_key_id=..., aws_secret_access_key=..., region_name=...

In [ ]:
#@title Model availabitity etc
#@markdown `On demand` means you can use it.
# Initialize the boto3 client for Bedrock
bedrock_client = boto3.client(
    'bedrock',
    aws_access_key_id=credentials['aws_access_key_id'],
    aws_secret_access_key=credentials['aws_secret_access_key'],
    region_name=credentials['aws_region']
)
bedrock_client.list_foundation_models()['modelSummaries']

In [ ]:
#Create a Bedrock client using the parsed credentials

import boto3
import json
import pandas as pd
from botocore.config import Config
import time


bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    aws_access_key_id=credentials['aws_access_key_id'],
    aws_secret_access_key=credentials['aws_secret_access_key'],
    region_name=credentials['aws_region'],
    config=Config(retries={"max_attempts": 10, "mode": "adaptive"})
)

In [ ]:
import re

def extract_json_from_model_output(text: str):
    """
   Extracts the first valid JSON object from LLM output.
   """

    if not text or not isinstance(text, str):
        return None

    # --- 1. Remove common reasoning wrappers ---
    text = re.sub(r"<tool_call>.*?<tool_call>", "", text, flags=re.DOTALL)
    text = re.sub(r"```json", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text)

    # --- 2. Find all JSON-like blocks ---
    candidates = re.findall(r"\{[\s\S]*?\}", text)

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except Exception:
            continue

    return None


In [ ]:
import json
import time


for idx, row in cluster_metadata_df.iterrows():

    cluster_id = int(row["Cluster_ID"])
    reps = row["Top_Paragraphs"]
    terms = row["Top_TFIDF_Terms"]

    reps_text = "\n\n".join(
        [f"Document {i+1}:\n\"\"\"{p}\"\"\"" for i, p in enumerate(reps)]
    )

    prompt = f'''

Act like an expert political scientist specializing in Greek parliamentary discourse and topic labeling of parliamentary debate clusters.

Your goal is to assign a short, accurate, and specific 3–6 word English title to the main thematic cluster shared by the texts below, and to mark whether it relates to the Tempi train crash.

Task:
Read the inputs, infer the dominant theme of the cluster, and output a JSON object with a single key "title" whose value is the final cluster title.

Context:
Representative speeches from this cluster:
<<<SPEECHES_START
{reps_text}
SPEECHES_END>>>

Top TF-IDF terms:
<<<TERMS_START
{terms}
TERMS_END>>>

Clarifications about Tempi:
The phrase “Tempi train crash” refers to the fatal train collision near Tempi, Greece, on 28 February 2023 and its political, legal, and institutional aftermath (e.g. rail safety, investigations, accountability, compensation, infrastructure, ministerial responsibility).

Step-by-step instructions:
1) Carefully read the representative speeches and TF-IDF terms.
2) Identify the single main political theme shared by most documents (e.g. policy area, scandal, institution, crisis, legislative initiative).
3) If multiple topics appear, choose the most dominant one, not “miscellaneous” or “mixed topics”.
4) Write a concise, informative 3–6 word English title that a political scientist could understand without context. Do not mention “cluster”, “topic”, or “speeches” in the title.
5) If the main theme is directly or indirectly about the Tempi train crash or its consequences (including rail safety, investigations, responsibilities, reforms), append exactly one asterisk character * at the very end of the title with no extra spaces.
6) Ensure the output is valid JSON with double quotes and no trailing commas, in exactly this shape:
{{"title": "your title here"}}

Constraints:
- Language: English only.
- Length: 3–6 words (not fewer, not more).
- Format: Output must be a single JSON object with exactly one key "title" and nothing else. No explanations, no reasoning, no extra keys, no surrounding text.

Take a deep breath and work on this problem step-by-step.
'''

    #Send to DeepSeek R1 Instruct via Bedrock
    try:
        response = bedrock_runtime.converse(
            modelId="us.deepseek.r1-v1:0",
            messages=[
                 {"role": "user", "content": [{"text": prompt}]}
                 ]
                 )
        content_blocks = response["output"]["message"]["content"]
        model_reply = "".join(
             block.get("text", "")
             for block in content_blocks
         )
        result = extract_json_from_model_output(model_reply)
        if result:
            title = result.get("title")
        
        else:
            title = None
        
        cluster_metadata_df.at[idx, "Cluster_title"] = title
        print(
            f"Row {idx+1} annotated | "
            f"title: {title}"
        )
    except Exception as e:
        print(f"Error on row {idx+1}: {e}")
        continue

    except Exception as e:
        print(f"Error on row {idx+1}: {e}")
        continue

time.sleep(0.5)  # polite pacing

cluster_metadata_df.to_csv("./out_files/02_clusters_with_titles.csv",
          index=False, encoding="utf-8-sig")


print("Saved: 02_clusters_with_titles.csv")


## Cluster Plots

In [ ]:
import pandas as pd
df_clusters = pd.read_csv("./out_files/02_clusters_with_titles.csv", encoding='utf-8-sig')

In [ ]:
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

os.makedirs("./out_files/plots/clustering/", exist_ok=True)

# Rename columns to match what the original code expects for the lookups
df_clusters = df_clusters.rename(columns={
    'Cluster_ID': 'cluster_id',
    'Cluster_title': 'cluster_title',
    'Doc_Count': 'num_documents'
})

# Create df_top_terms
records = []
for idx, row in df_clusters.iterrows():
    cluster_id = row['cluster_id']
    terms_str = row['Top_TFIDF_Terms']
    if pd.isna(terms_str):
        continue
    # terms_str looks like: "νομοσχεδιο εχει (0.196), νομοσχεδιο ενα (0.176)"
    # We can split by comma
    terms_list = terms_str.split(',')
    for term_entry in terms_list:
        term_entry = term_entry.strip()
        if not term_entry: continue
        # Extract term and tfidf
        match = re.match(r'(.*)\s+\(([\d\.]+)\)', term_entry)
        if match:
            term = match.group(1).strip()
            tfidf = float(match.group(2))
            records.append({'cluster': cluster_id, 'term': term, 'tfidf': tfidf})

df_top_terms = pd.DataFrame(records)

# --- Updated Logic for Specific Clusters ---
target_clusters = [7, 8, 35, 29, 41]
top_n_plot = 10

# A 2x3 grid gives us 6 slots, perfect for 5 clusters (1 will be left empty)
cols = 3
rows = 2

print(f"Rendering single page for clusters: {target_clusters}")

# Increased figure height to accommodate 2 rows
fig, axes = plt.subplots(rows, cols, figsize=(16, 10)) 
axes = axes.flatten()

for ax_idx, cluster_id in enumerate(target_clusters):
    ax = axes[ax_idx]

    # Select TF-IDF rows
    data = (
        df_top_terms[df_top_terms["cluster"] == cluster_id]
        .sort_values("tfidf", ascending=False)
        .head(top_n_plot)
    )

    # Fetch cluster details safely
    num_docs_raw = df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "num_documents"].values
    title_raw = df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "cluster_title"].values
    
    # Check if the cluster exists in the dataframe before attempting to plot
    if len(num_docs_raw) == 0:
        ax.set_title(f"Cluster {cluster_id} not found", fontsize=14)
        continue
        
    num_docs = int(num_docs_raw[0])
    title_txt = f'{title_raw[0]}\n({num_docs} par.)'

    sns.barplot(
        data=data,
        x="tfidf",
        y="term",
        hue="term",
        legend=False,
        palette="Blues_r",
        orient='h',
        ax=ax
    )

    ax.set_title(f"{cluster_id}: {title_txt}", fontsize=14)
    ax.set_xlabel("TF-IDF", fontsize=11)
    ax.set_ylabel("")
    ax.tick_params(axis='x', labelrotation=45, labelsize=9)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_xlim(0, 1)

# Turn off remaining empty axes (the 6th slot)
for j in range(len(target_clusters), len(axes)):
    axes[j].axis("off")

fig.suptitle(
    "Top 10 TF-IDF Terms for Selected Clusters",
    fontsize=22,
    y=1.02 
)

plt.tight_layout()

# Updated output filename to reflect the custom selection
out_path = './out_files/plots/clustering/02_top10_TF-IDF_selected_clusters.jpg'
plt.savefig(out_path, dpi=500, bbox_inches='tight')
plt.close()

print(f"Plot successfully generated and saved to: {out_path}")

In [ ]:
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

os.makedirs("./out_files/plots/clustering/", exist_ok=True)


# Rename columns to match what the original code expects for the lookups
df_clusters = df_clusters.rename(columns={
    'Cluster_ID': 'cluster_id',
    'Cluster_title': 'cluster_title',
    'Doc_Count': 'num_documents'
})

# Create df_top_terms
records = []
for idx, row in df_clusters.iterrows():
    cluster_id = row['cluster_id']
    terms_str = row['Top_TFIDF_Terms']
    if pd.isna(terms_str):
        continue
    # terms_str looks like: "νομοσχεδιο εχει (0.196), νομοσχεδιο ενα (0.176)"
    # We can split by comma
    terms_list = terms_str.split(',')
    for term_entry in terms_list:
        term_entry = term_entry.strip()
        if not term_entry: continue
        # Extract term and tfidf
        match = re.match(r'(.*)\s+\(([\d\.]+)\)', term_entry)
        if match:
            term = match.group(1).strip()
            tfidf = float(match.group(2))
            records.append({'cluster': cluster_id, 'term': term, 'tfidf': tfidf})

df_top_terms = pd.DataFrame(records)

# Now, we define the variables needed for the plot loop
cluster_counts = df_clusters['cluster_id'].unique()
top_n_plot = 10
clusters_per_page = 3
cols = 3
rows = 1

num_clusters = len(cluster_counts)
num_pages = math.ceil(num_clusters / clusters_per_page)

print(f"Total clusters: {num_clusters}")
print(f"Pages to generate: {num_pages}")

for page in range(num_pages):
    start_idx = page * clusters_per_page
    end_idx = min(start_idx + clusters_per_page, num_clusters)
    cluster_range = list(range(start_idx, end_idx))

    print(f"Rendering page {page+1}: clusters {cluster_range}")

    fig, axes = plt.subplots(rows, cols, figsize=(16, 5)) 
    axes = axes.flatten()

    for ax_idx, cluster_id in enumerate(cluster_range):
        ax = axes[ax_idx]

        # Select TF-IDF rows
        data = (
            df_top_terms[df_top_terms["cluster"] == cluster_id]
            .sort_values("tfidf", ascending=False)
            .head(top_n_plot)
        )

        num_docs = int(df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "num_documents"].values[0])
        title_txt = f'{df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "cluster_title"].values[0]}\n({num_docs} par.)'

        sns.barplot(
            data=data,
            x="tfidf",
            y="term",
            hue="term",
            legend=False,
            palette="Blues_r",
            orient='h',
            ax=ax
        )

        ax.set_title(f"{cluster_id}: {title_txt}", fontsize=14)
        ax.set_xlabel("TF-IDF", fontsize=11)
        ax.set_ylabel("")
        ax.tick_params(axis='x', labelrotation=45, labelsize=9)
        ax.tick_params(axis='y', labelsize=11)
        ax.set_xlim(0, 1)

    # Turn off remaining empty axes
    for j in range(len(cluster_range), len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        f"Top 10 TF-IDF Terms per Cluster\nPage {page+1} of {num_pages}",
        fontsize=22,
        y=1.05 
    )

    plt.tight_layout()

    out_path = f'./out_files/plots/clustering/02_top10_TF-IDF_page_{page+1}.jpg'
    plt.savefig(out_path, dpi=500, bbox_inches='tight')
    plt.close()

print("Plots successfully generated.")

In [ ]:
kmeans_dataset = pd.read_csv('./out_files/02_dataset_clustered.csv', encoding='utf-8-sig')
kmeans_dataset.info()

In [ ]:
df_clusters.info()

In [ ]:

# --- Compute absolute counts ---
cluster_party_counts = (
    kmeans_dataset
    .groupby(["cluster_label", "party"])
    .size()
    .reset_index(name="speech_count")
)

# --- Compute % within each cluster ---
cluster_party_counts["percent_within_cluster"] = (
    cluster_party_counts.groupby("cluster_label")["speech_count"]
    .transform(lambda x: x / x.sum() * 100)
)
cluster_party_counts = cluster_party_counts[
    cluster_party_counts["cluster_label"].isin(kmeans_dataset["cluster_label"].unique())
]


# --- Define color palette ---
party_colors = {
    "Nea Dimokratia": "#1f77b4",
    "SYRIZA-PS": "#e377c2",
    "Nea Aristera": "#c2185b",
    "PASOK-KINAL": "#2ca02c",
    "Plefsi Eleftherias": "#9467bd",
    "KKE": "#d62728",
    "Elliniki Lysi": "#17becf",
    "Niki": "#003366",
    "Spartiates": "#000000",
    "Independents": "#ffeb3b",
    "MeRA25": "#ff7f0e"
}

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os

# Create the output directory if it doesn't exist
os.makedirs("./out_files/plots/clustering/", exist_ok=True)

# -------------------------------------------------------------------
# Prepare data
# -------------------------------------------------------------------
party_totals = kmeans_dataset.groupby("party").size().rename("party_total")

df1 = cluster_party_counts.merge(party_totals, on="party")
df1["pct_of_party"] = df1["speech_count"] / df1["party_total"]

# FIX: Changed "cluster" to "cluster_label"
num_clusters = df1["cluster_label"].nunique()

# -------------------------------------------------------------------
# Paging configuration (Updated for 3 per page)
# -------------------------------------------------------------------
clusters_per_page = 3
cols = 3
rows = 1
num_pages = math.ceil(num_clusters / clusters_per_page)

print(f"Total clusters: {num_clusters}")
print(f"Pages to generate: {num_pages}")

# -------------------------------------------------------------------
# Loop pages
# -------------------------------------------------------------------
for page in range(num_pages):

    start_idx = page * clusters_per_page
    end_idx = min(start_idx + clusters_per_page, num_clusters)
    cluster_range = list(range(start_idx, end_idx))

    print(f"Rendering page {page+1}: clusters {cluster_range}")

    # Create figure (Adjusted height to 5 for a 1x3 grid)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5))
    
    # Ensure axes is flattenable correctly
    if type(axes) not in [list, tuple] and not isinstance(axes, np.ndarray):
        axes = [axes]
    elif isinstance(axes, np.ndarray):
        axes = axes.flatten()

    # -------------------------------------------------------------------
    # Plot each cluster within this page
    # -------------------------------------------------------------------
    for ax_idx, cluster_id in enumerate(cluster_range):

        ax = axes[ax_idx]

        # FIX: Changed "cluster" to "cluster_label"
        data = (
            df1[df1["cluster_label"] == cluster_id]
            .sort_values("pct_of_party", ascending=False)
        )

        try:
            title = df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "cluster_title"].values[0]
            num_docs = int(df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "num_documents"].values[0])
        except:
            title, num_docs = f"Cluster {cluster_id}", 0

        sns.barplot(
            data=data,
            x="pct_of_party",
            y="party",
            hue="party",
            orient="h",
            dodge=False,
            palette=party_colors,
            ax=ax
        )
        
        # Remove the legend safely
        if ax.get_legend() is not None:
            ax.get_legend().remove()

        ax.set_title(f"{cluster_id}: {title}\n({num_docs} par.)", fontsize=14)
        ax.set_xlabel("% of party's paragraphs")
        ax.set_ylabel("")
        ax.tick_params(axis='x', labelrotation=45)

        max_val = data["pct_of_party"].max()
        ax.set_xlim(0, max_val * 1.1 if max_val > 0 else 0.05)

    # Turn off extra axes
    for j in range(len(cluster_range), len(axes)):
        axes[j].axis("off")

    # Page title (Anchored directly above the 1-row layout)
    fig.suptitle(
        f"Percentage of Each Party's Paragraphs in Each Cluster\nPage {page+1} of {num_pages}",
        fontsize=22,
        y=1.12
    )

    # Layout
    plt.tight_layout()

    # Save
    out_path = f"./out_files/plots/clustering/03_perc_of_party_page_{page+1}.jpg"
    plt.savefig(out_path, dpi=500, bbox_inches='tight')
    plt.close()

print("Plots successfully generated.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os
import numpy as np
import pandas as pd

# Ensure the output directory exists
os.makedirs("./out_files/plots/clustering/", exist_ok=True)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------
total_speeches = kmeans_dataset.shape[0]
party_totals = kmeans_dataset.groupby("party").size().rename("party_total")
party_global_share = (party_totals / total_speeches).rename("party_global_share")

df2 = cluster_party_counts.merge(party_totals, on="party")
df2 = df2.merge(party_global_share, on="party")

# FIX: Changed "cluster" to "cluster_label" to match kmeans_dataset schema
cluster_totals = cluster_party_counts.groupby("cluster_label")["speech_count"].sum().rename("cluster_total")
df2 = df2.merge(cluster_totals, on="cluster_label")

df2["cluster_share"] = df2["speech_count"] / df2["cluster_total"]
df2["representation_deviation"] = df2["cluster_share"] - df2["party_global_share"]

# FIX: Changed "cluster" to "cluster_label"
num_clusters = df2["cluster_label"].nunique()

# ------------------------------------------------------------
# Paging configuration (Updated for 3 per page)
# ------------------------------------------------------------
clusters_per_page = 3  # Reduced from 6
cols = 3
rows = 1               # Reduced from 2
num_pages = math.ceil(num_clusters / clusters_per_page)

print(f"Total clusters: {num_clusters}")
print(f"Pages to generate: {num_pages}")

# ------------------------------------------------------------
# Page Loop
# ------------------------------------------------------------
for page in range(num_pages):

    start_idx = page * clusters_per_page
    end_idx = min(start_idx + clusters_per_page, num_clusters)
    cluster_range = list(range(start_idx, end_idx))

    print(f"Rendering page {page+1}: clusters {cluster_range}")

    # Create figure (Adjusted height to 5 for a 1x3 grid)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5))
    
    # Ensure axes is flattenable correctly
    if type(axes) not in [list, tuple] and not isinstance(axes, np.ndarray):
        axes = [axes]
    elif isinstance(axes, np.ndarray):
        axes = axes.flatten()

    for ax_idx, cluster_id in enumerate(cluster_range):

        ax = axes[ax_idx]

        # FIX: Changed "cluster" to "cluster_label"
        data = (
            df2[df2["cluster_label"] == cluster_id]
            .sort_values("representation_deviation", ascending=False)
        )

        try:
            title = df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "cluster_title"].values[0]
            num_docs = int(df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "num_documents"].values[0])
        except:
            title, num_docs = f"Cluster {cluster_id}", 0

        sns.barplot(
            data=data,
            x="representation_deviation",
            y="party",
            hue="party",
            orient="h",
            dodge=False,
            palette=party_colors,
            ax=ax
        )
        
        # Remove the legend safely
        if ax.get_legend() is not None:
            ax.get_legend().remove()

        # Zero-line (neutral point)
        ax.axvline(0, color="black", linewidth=1)

        ax.set_title(f"{cluster_id}: {title}\n({num_docs} par.)", fontsize=14)
        ax.set_xlabel("Over / Under Representation")
        ax.set_ylabel("")
        ax.tick_params(axis='x', labelrotation=45)

        # Auto symmetric limits (looks cleaner)
        max_dev = abs(data["representation_deviation"]).max()
        # Fallback in case max_dev is exactly 0 or NaN
        if pd.isna(max_dev) or max_dev == 0:
            max_dev = 0.1 
        ax.set_xlim(-max_dev * 1.1, max_dev * 1.1)

    # Turn off empty axes
    for j in range(len(cluster_range), len(axes)):
        axes[j].axis("off")

    # Suptitle (Anchored directly above the 1-row layout)
    fig.suptitle(
        f"Party Representation Deviation Across Clusters\nPage {page+1} of {num_pages}",
        fontsize=22,
        y=1.12
    )

    # Adjust layout
    plt.tight_layout()

    # Save output page
    out_path = f"./out_files/plots/clustering/04_representation_deviation_page_{page+1}.jpg"
    plt.savefig(out_path, dpi=500, bbox_inches='tight')
    plt.close()

print("DONE.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os
import numpy as np
import pandas as pd

# Ensure the output directory exists
os.makedirs("./out_files/plots/clustering/", exist_ok=True)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------
# FIX: Changed "cluster" to "cluster_label"
cluster_totals = (
    cluster_party_counts.groupby("cluster_label")["speech_count"]
    .sum()
    .rename("cluster_total")
)

# FIX: Changed "cluster" to "cluster_label"
df3 = cluster_party_counts.merge(cluster_totals, on="cluster_label")
df3["pct_within_cluster"] = df3["speech_count"] / df3["cluster_total"]

# FIX: Changed "cluster" to "cluster_label"
num_clusters = df3["cluster_label"].nunique()

# ------------------------------------------------------------
# Paging configuration (Updated for 3 per page)
# ------------------------------------------------------------
clusters_per_page = 3  # Reduced from 6
cols = 3
rows = 1               # Reduced from 2

num_pages = math.ceil(num_clusters / clusters_per_page)

print(f"Total clusters: {num_clusters}")
print(f"Pages to generate: {num_pages}")

# ------------------------------------------------------------
# Page Loop
# ------------------------------------------------------------
for page in range(num_pages):

    start_idx = page * clusters_per_page
    end_idx = min(start_idx + clusters_per_page, num_clusters)
    cluster_range = list(range(start_idx, end_idx))

    print(f"Rendering page {page + 1}: clusters {cluster_range}")

    # Create figure (Adjusted height to 5 for a 1x3 grid)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5))
    
    # Ensure axes is flattenable correctly
    if type(axes) not in [list, tuple] and not isinstance(axes, np.ndarray):
        axes = [axes]
    elif isinstance(axes, np.ndarray):
        axes = axes.flatten()

    for ax_idx, cluster_id in enumerate(cluster_range):

        ax = axes[ax_idx]

        # FIX: Changed "cluster" to "cluster_label"
        data = (
            df3[df3["cluster_label"] == cluster_id]
            .sort_values("pct_within_cluster", ascending=False)
        )

        try:
            # df_clusters correctly uses "cluster_id"
            title = df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "cluster_title"].values[0]
            num_docs = int(df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "num_documents"].values[0])
        except:
            title, num_docs = f"Cluster {cluster_id}", 0

        sns.barplot(
            data=data,
            x="pct_within_cluster",
            y="party",
            hue="party",
            orient="h",
            dodge=False,
            palette=party_colors,
            ax=ax
        )
        
        # Remove the legend safely
        if ax.get_legend() is not None:
            ax.get_legend().remove()

        # Changed to par.
        ax.set_title(f"{cluster_id}: {title}\n({num_docs} par.)", fontsize=14)
        ax.set_xlabel("% of cluster (party composition)")
        ax.set_ylabel("")
        ax.tick_params(axis="x", labelrotation=45)

        # Expand the x-axis slightly for visual clarity
        max_val = data["pct_within_cluster"].max()
        ax.set_xlim(0, max_val * 1.1 if max_val > 0 else 0.05)

    # Turn off any leftover empty axes
    for j in range(len(cluster_range), len(axes)):
        axes[j].axis("off")

    # Page title (Anchored directly above the 1-row layout)
    fig.suptitle(
        f"Party Composition Within Each Cluster\nPage {page + 1} of {num_pages}",
        fontsize=22,
        y=1.12
    )

    # Layout
    plt.tight_layout()

    # Save page
    out_path = f"./out_files/plots/clustering/05_party_composition_page_{page + 1}.jpg"
    plt.savefig(out_path, dpi=500, bbox_inches="tight")
    plt.close()

print("DONE.")

## Tempi Plots

In [ ]:
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import textwrap

os.makedirs("./out_files/plots/clustering/", exist_ok=True)

# Rename columns to match what the original code expects for the lookups
df_clusters = df_clusters.rename(columns={
    'Cluster_ID': 'cluster_id',
    'Cluster_title': 'cluster_title',
    'Doc_Count': 'num_documents'
})

# Create df_top_terms
records = []
for idx, row in df_clusters.iterrows():
    cluster_id = row['cluster_id']
    terms_str = row['Top_TFIDF_Terms']
    if pd.isna(terms_str):
        continue

    terms_list = terms_str.split(',')
    for term_entry in terms_list:
        term_entry = term_entry.strip()
        if not term_entry: continue
        # Extract term and tfidf
        match = re.match(r'(.*)\s+\(([\d\.]+)\)', term_entry)
        if match:
            term = match.group(1).strip()
            tfidf = float(match.group(2))
            records.append({'cluster': cluster_id, 'term': term, 'tfidf': tfidf})

df_top_terms = pd.DataFrame(records)

# --- Updated Logic for Specific Clusters ---
target_clusters = [7, 8, 35, 39, 41]
top_n_plot = 10

# A 2x3 grid gives us 6 slots, perfect for 5 clusters (1 will be left empty)
cols = 3
rows = 2

print(f"Rendering single page for clusters: {target_clusters}")

# Increased figure height to accommodate 2 rows
fig, axes = plt.subplots(rows, cols, figsize=(18, 10)) 
axes = axes.flatten()

for ax_idx, cluster_id in enumerate(target_clusters):
    ax = axes[ax_idx]

    # Select TF-IDF rows
    data = (
        df_top_terms[df_top_terms["cluster"] == cluster_id]
        .sort_values("tfidf", ascending=False)
        .head(top_n_plot)
    )

    # Fetch cluster details safely
    num_docs_raw = df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "num_documents"].values
    title_raw = df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "cluster_title"].values
    
    # Check if the cluster exists in the dataframe before attempting to plot
    if len(num_docs_raw) == 0:
        ax.set_title(f"Cluster {cluster_id} not found", fontsize=14)
        continue
        
    num_docs = int(num_docs_raw[0])

    # 1. Get the raw title string
    raw_title_string = str(title_raw[0])
    
    # 2. Wrap the text (adjust 'width=30' to be wider or narrower as needed)
    wrapped_title = textwrap.fill(raw_title_string, width=40)
    
    # 3. Combine the wrapped title with your paragraph count
    title_txt = f'{wrapped_title}\n({num_docs} par.)'


    sns.barplot(
        data=data,
        x="tfidf",
        y="term",
        hue="term",
        legend=False,
        palette="Blues_r",
        orient='h',
        ax=ax
    )

    ax.set_title(f"{cluster_id}: {title_txt}", fontsize=16)
    ax.set_xlabel("TF-IDF", fontsize=12)
    ax.set_ylabel("")
    ax.tick_params(axis='x', labelrotation=45, labelsize=10)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_xlim(0, 1)

# Turn off remaining empty axes (the 6th slot)
for j in range(len(target_clusters), len(axes)):
    axes[j].axis("off")

plt.tight_layout(h_pad=3.0)

# Updated output filename to reflect the custom selection
out_path = './out_files/plots/clustering/01_top10_TF-IDF_tempi.jpg'
plt.savefig(out_path, dpi=500, bbox_inches='tight')
plt.close()

print(f"Plot successfully generated and saved to: {out_path}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os
import textwrap

# Create the output directory if it doesn't exist
os.makedirs("./out_files/plots/clustering/", exist_ok=True)

# -------------------------------------------------------------------
# Prepare data
# -------------------------------------------------------------------
party_totals = kmeans_dataset.groupby("party").size().rename("party_total")

df1 = cluster_party_counts.merge(party_totals, on="party")
df1["pct_of_party"] = df1["speech_count"] / df1["party_total"]

# -------------------------------------------------------------------
# Target Clusters Configuration 
# -------------------------------------------------------------------
target_clusters = [7, 8, 35, 39, 41]

# A 2x3 grid gives us 6 slots, perfect for 5 clusters (1 will be left empty)
cols = 3
rows = 2

print(f"Rendering single page for clusters: {target_clusters}")

# Create figure (Adjusted width to 18, height to 10 for a 2x3 grid)
fig, axes = plt.subplots(rows, cols, figsize=(18, 10))

# Ensure axes is flattenable correctly
if type(axes) not in [list, tuple] and not isinstance(axes, np.ndarray):
    axes = [axes]
elif isinstance(axes, np.ndarray):
    axes = axes.flatten()

# -------------------------------------------------------------------
# Plot each target cluster
# -------------------------------------------------------------------
for ax_idx, cluster_id in enumerate(target_clusters):

    ax = axes[ax_idx]

    data = (
        df1[df1["cluster_label"] == cluster_id]
        .sort_values("pct_of_party", ascending=False)
    )

    try:
        title_raw = str(df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "cluster_title"].values[0])
        num_docs = int(df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "num_documents"].values[0])
        # Wrap the text title
        title = textwrap.fill(title_raw, width=40)
    except:
        title, num_docs = f"Cluster {cluster_id}", 0

    # Ensure the dataframe isn't empty for this cluster before plotting
    if not data.empty:
        sns.barplot(
            data=data,
            x="pct_of_party",
            y="party",
            hue="party",
            orient="h",
            dodge=False,
            palette=party_colors,
            ax=ax
        )
    else:
        ax.text(0.5, 0.5, "No Data", ha='center', va='center')

    # Remove the legend safely
    if ax.get_legend() is not None:
        ax.get_legend().remove()

    # Apply updated font sizes
    ax.set_title(f"{cluster_id}: {title}\n({num_docs} par.)", fontsize=16)
    ax.set_xlabel("% of party's paragraphs", fontsize=12)
    ax.set_ylabel("")
    ax.tick_params(axis='x', labelrotation=45, labelsize=10)
    ax.tick_params(axis='y', labelsize=14)

    max_val = data["pct_of_party"].max() if not data.empty else 0
    ax.set_xlim(0, max_val * 1.1 if max_val > 0 else 0.05)

# Turn off extra axes (the 6th slot)
for j in range(len(target_clusters), len(axes)):
    axes[j].axis("off")


# Layout
plt.tight_layout(h_pad=3.0)

# Save
out_path = "./out_files/plots/clustering/01_perc_of_party_tempi.jpg"
plt.savefig(out_path, dpi=500, bbox_inches='tight')
plt.close()

print(f"Plot successfully generated and saved to: {out_path}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os
import numpy as np
import pandas as pd

# Ensure the output directory exists
os.makedirs("./out_files/plots/clustering/", exist_ok=True)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------
total_speeches = kmeans_dataset.shape[0]
party_totals = kmeans_dataset.groupby("party").size().rename("party_total")
party_global_share = (party_totals / total_speeches).rename("party_global_share")

df2 = cluster_party_counts.merge(party_totals, on="party")
df2 = df2.merge(party_global_share, on="party")

# FIX: Changed "cluster" to "cluster_label" to match kmeans_dataset schema
cluster_totals = cluster_party_counts.groupby("cluster_label")["speech_count"].sum().rename("cluster_total")
df2 = df2.merge(cluster_totals, on="cluster_label")

df2["cluster_share"] = df2["speech_count"] / df2["cluster_total"]
df2["representation_deviation"] = df2["cluster_share"] - df2["party_global_share"]

# ------------------------------------------------------------
# Target Clusters Configuration
# ------------------------------------------------------------
target_clusters = [7, 8, 35, 39, 41]

# A 2x3 grid gives us 6 slots, perfect for 5 clusters (1 will be left empty)
cols = 3
rows = 2

print(f"Rendering single page for clusters: {target_clusters}")

# Create figure (Adjusted height to 10 for a 2x3 grid)
fig, axes = plt.subplots(rows, cols, figsize=(16, 10))

# Ensure axes is flattenable correctly
if type(axes) not in [list, tuple] and not isinstance(axes, np.ndarray):
    axes = [axes]
elif isinstance(axes, np.ndarray):
    axes = axes.flatten()

# ------------------------------------------------------------
# Plot each target cluster
# ------------------------------------------------------------
for ax_idx, cluster_id in enumerate(target_clusters):

    ax = axes[ax_idx]

    # FIX: Changed "cluster" to "cluster_label"
    data = (
        df2[df2["cluster_label"] == cluster_id]
        .sort_values("representation_deviation", ascending=False)
    )

    try:
        title = df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "cluster_title"].values[0]
        num_docs = int(df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "num_documents"].values[0])
    except:
        title, num_docs = f"Cluster {cluster_id}", 0

    if not data.empty:
        sns.barplot(
            data=data,
            x="representation_deviation",
            y="party",
            hue="party",
            orient="h",
            dodge=False,
            palette=party_colors,
            ax=ax
        )
    else:
        ax.text(0.5, 0.5, "No Data", ha='center', va='center')

    # Remove the legend safely
    if ax.get_legend() is not None:
        ax.get_legend().remove()

    # Zero-line (neutral point)
    ax.axvline(0, color="black", linewidth=1)

    ax.set_title(f"{cluster_id}: {title}\n({num_docs} par.)", fontsize=14)
    ax.set_xlabel("Over / Under Representation")
    ax.set_ylabel("")
    ax.tick_params(axis='x', labelrotation=45)

    # Auto symmetric limits (looks cleaner)
    max_dev = abs(data["representation_deviation"]).max() if not data.empty else 0
    # Fallback in case max_dev is exactly 0 or NaN
    if pd.isna(max_dev) or max_dev == 0:
        max_dev = 0.1 
    ax.set_xlim(-max_dev * 1.1, max_dev * 1.1)

# Turn off empty axes (the 6th slot)
for j in range(len(target_clusters), len(axes)):
    axes[j].axis("off")


# Adjust layout
plt.tight_layout(h_pad=3.0)

# Save output page
out_path = "./out_files/plots/clustering/01_representation_deviation_tempi.jpg"
plt.savefig(out_path, dpi=500, bbox_inches='tight')
plt.close()

print(f"Plot successfully generated and saved to: {out_path}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os
import numpy as np
import pandas as pd
import textwrap

# Ensure the output directory exists
os.makedirs("./out_files/plots/clustering/", exist_ok=True)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------
# FIX: Changed "cluster" to "cluster_label"
cluster_totals = (
    cluster_party_counts.groupby("cluster_label")["speech_count"]
    .sum()
    .rename("cluster_total")
)

# FIX: Changed "cluster" to "cluster_label"
df3 = cluster_party_counts.merge(cluster_totals, on="cluster_label")
df3["pct_within_cluster"] = df3["speech_count"] / df3["cluster_total"]

# ------------------------------------------------------------
# Target Clusters Configuration
# ------------------------------------------------------------
target_clusters = [7, 8, 35, 39, 41]

# A 2x3 grid gives us 6 slots, perfect for 5 clusters (1 will be left empty)
cols = 3
rows = 2

print(f"Rendering single page for clusters: {target_clusters}")

# Create figure (Adjusted width to 18, height to 10 for a 2x3 grid)
fig, axes = plt.subplots(rows, cols, figsize=(18, 10))

# Ensure axes is flattenable correctly
if type(axes) not in [list, tuple] and not isinstance(axes, np.ndarray):
    axes = [axes]
elif isinstance(axes, np.ndarray):
    axes = axes.flatten()

# ------------------------------------------------------------
# Plot each target cluster
# ------------------------------------------------------------
for ax_idx, cluster_id in enumerate(target_clusters):

    ax = axes[ax_idx]

    # FIX: Changed "cluster" to "cluster_label"
    data = (
        df3[df3["cluster_label"] == cluster_id]
        .sort_values("pct_within_cluster", ascending=False)
    )

    try:
        # df_clusters correctly uses "cluster_id"
        title_raw = str(df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "cluster_title"].values[0])
        num_docs = int(df_clusters.loc[df_clusters["cluster_id"] == cluster_id, "num_documents"].values[0])
        # Wrap the text title
        title = textwrap.fill(title_raw, width=40)
    except:
        title, num_docs = f"Cluster {cluster_id}", 0

    if not data.empty:
        sns.barplot(
            data=data,
            x="pct_within_cluster",
            y="party",
            hue="party",
            orient="h",
            dodge=False,
            palette=party_colors,
            ax=ax
        )
    else:
        ax.text(0.5, 0.5, "No Data", ha='center', va='center')

    # Remove the legend safely
    if ax.get_legend() is not None:
        ax.get_legend().remove()

    # Apply updated font sizes and wrap
    ax.set_title(f"{cluster_id}: {title}\n({num_docs} par.)", fontsize=16)
    ax.set_xlabel("% of cluster (party composition)", fontsize=12)
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelrotation=45, labelsize=10)
    ax.tick_params(axis="y", labelsize=14)

    # Expand the x-axis slightly for visual clarity
    max_val = data["pct_within_cluster"].max() if not data.empty else 0
    ax.set_xlim(0, max_val * 1.1 if max_val > 0 else 0.05)

# Turn off any leftover empty axes (the 6th slot)
for j in range(len(target_clusters), len(axes)):
    axes[j].axis("off")


# Layout
plt.tight_layout(h_pad=3.0)

# Save page
out_path = "./out_files/plots/clustering/01_party_composition_tempi.jpg"
plt.savefig(out_path, dpi=500, bbox_inches="tight")
plt.close()

print(f"Plot successfully generated and saved to: {out_path}")

# Tempi_df - Narrative Extraction

In [ ]:
df_pars = pd.read_csv('./out_files/02_dataset_clustered.csv', encoding='utf-8-sig')
df_clusters = pd.read_csv('./out_files/02_clusters_with_titles.csv', encoding='utf-8-sig')

In [ ]:
tempi_clusters = (
    df_clusters.loc[
        df_clusters["Cluster_title"].str.contains(r"\*", regex=True),
        "Cluster_ID"
    ]
    .tolist()
)
print("Clusters related to Tempi train crash:", tempi_clusters)

In [ ]:
tempi_df = df_pars.loc[df_pars['cluster_label'].isin(tempi_clusters)].copy()

# Keep original index as a column
tempi_df = tempi_df.reset_index()
tempi_df = tempi_df.rename(columns={'index': 'original_index'})

tempi_df.to_csv('./out_files/03_tempi_df_dataset.csv', index=False, encoding='utf-8-sig')


In [ ]:
tempi_df = pd.read_csv('./out_files/03_tempi_df_dataset.csv', encoding='utf-8-sig')
tempi_df.info()

## Extract Narratives

* Using the same Client as before

In [ ]:
import json
import pandas as pd
from botocore.config import Config
import time

# --- Load dataset ---
df = pd.read_csv("./out_files/03_tempi_df_dataset.csv", encoding="utf-8-sig")

# Add output column
df["narrative"] = ""

# --- Main loop ---
for idx, row in df.iterrows():
    sitting_date = row.get("sitting_date", "")
    political_party = row.get("party", "")
    paragraph = row.get("paragraph", "")

    # --- Prompt construction ---
    prompt = f"""

Act like a senior political discourse analyst and annotation specialist with expertise in narrative extraction and framing analysis.

Your goal is to identify and distill a political narrative—defined as a recurring, specific overt or implicit claim that promotes a clear interpretation of an ongoing political issue and functions as both a cognitive and strategic tool—into a single, highly precise sentence.

Task: Read one paragraph from a Greek parliamentary speech and output its core narrative about the Tempi train crash (Greece, 2023) or return "irrelevant".

Inputs you receive (per row):
- sitting_date: {sitting_date}
- political_party: {political_party}
- paragraph: {paragraph}

Requirements:
1) Work exclusively from the paragraph; use other fields only for minimal contextual clarification.
2) A valid narrative must reflect a recurring or generalizable claim with a clear, specific interpretation (e.g., concrete accountability, causal mechanism, or policy implication), not vague or generic statements.
3) Avoid abstract or broad wording; prefer concrete, content-rich phrasing tied to identifiable issues (e.g., safety failures, institutional responsibility, regulatory gaps).
4) If the paragraph does not substantively address the Tempi crash or its direct implications (accountability, investigation, safety, governance, legal consequences), output "irrelevant".
5) Output a neutral, declarative statement expressing the core narrative without attribution or speech-reporting verbs.
6) Do not include names, actors, quotes, slogans, rhetorical questions, or multiple sentences.
7) English only. Plain text. Maximum 8 words.

Process (apply sequentially):
1) Assess whether the paragraph contains a narrative related to the Tempi crash and its systemic or political implications.
2) If absent, classify as "irrelevant".
3) If present, identify the underlying recurring claim and its specific interpretation (cause, responsibility, or solution).
4) Reformulate it into a concise, concrete proposition capturing its strategic or cognitive function.
5) Eliminate vagueness by replacing general terms with specific mechanisms or implications.

Constraints:
- Format: JSON only
- Style: minimal, declarative, analytical, specific
- Scope: include only the core narrative claim;
- Reasoning: think step-by-step internally, but output only final result
- Self-check: ensure one sentence, ≤8 words, highly specific, no attribution, valid JSON

Output format (strict):
{{
  "narrative": "your one-sentence result here or the exact word irrelevant"
}}

Take a deep breath and work on this problem step-by-step.

"""

 #Send to DeepSeek R1 Instruct via Bedrock
    try:
      response = bedrock_runtime.converse(
          modelId="us.deepseek.r1-v1:0",
          messages=[
            {"role": "user", "content": [{"text": prompt}]}
        ]
      )

      model_reply = response["output"]["message"]["content"][0]["text"]

      # Try to parse JSON-like reply
      try:
        result = json.loads(model_reply)
      except Exception:
        result = {"narrative": "No valid JSON reply"}

      df.at[idx, "narrative"] = result.get("narrative", "")
      print(f"Row {idx+1} annotated | {result.get('narrative', '')}")

    except Exception as e:
      print(f"Error on row {idx+1}: {e}")
      continue


    time.sleep(0.3)  # polite pacing

# --- Save results ---
output_file = "./out_files/04_tempi_df_R1_narratives.csv"
df.to_csv(output_file, index=False)
print(f"\nAnnotation complete. Results saved to {output_file}")


## Clean Up Narratives

In [ ]:
import pandas as pd
narratives = pd.read_csv("./out_files/04_tempi_df_R1_narratives.csv", encoding='utf-8-sig')

In [ ]:
narratives.info()

narratives.head(2)

In [ ]:
# 1. Count how many duplicate IDs exist in the main dataframe
duplicate_count = narratives.duplicated(subset=['original_index']).sum()
print(f"Found {duplicate_count} duplicate rows based on 'original_index'.")

# 2. Isolate and display the duplicates so you can inspect them side-by-side
if duplicate_count > 0:
    # keep=False ensures we see BOTH the original and the duplicate row
    duplicate_rows = narratives[narratives.duplicated(subset=['original_index'], keep=False)]
    
    # Sort by the index so the duplicates are stacked right next to each other
    display(duplicate_rows.sort_values(by='original_index').head(10))

In [ ]:
import pandas as pd

# 1. Classify all entries
narratives["category"] = narratives["narrative"].apply(
    lambda x: x if x in ["irrelevant", "No valid JSON reply"] else "actual_narrative"
)

# 2. Group, count, and pivot
table_all = (
    narratives.groupby(["cluster_label", "category"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Define our columns early so we can use them for the sum
cols_all = ["actual_narrative", "irrelevant", "No valid JSON reply"]
existing_cols_all = [col for col in cols_all if col in table_all.columns]

# 3. Calculate total count per row 
# FIXED: We only sum the specific category columns, ignoring cluster_label
table_all["total_count"] = table_all[existing_cols_all].sum(axis=1)


# 3.5 Calculate the overall sum and append as a new row
numeric_cols_to_sum = ["total_count"] + existing_cols_all

# Calculate the sum for the numeric columns
total_sums = table_all[numeric_cols_to_sum].sum()
total_sums["cluster_label"] = "Total" # Label the row

# Append the new row to the bottom of the dataframe
table_all.loc[len(table_all)] = total_sums
# ---------------------------------------------------------

# 4. Format cells to show "Count (Percentage%)"
for col in existing_cols_all:
    count = table_all[col]
    pct = (count / table_all["total_count"] * 100).round(2)
    # Combine count and percentage into a single string
    table_all[col] = count.astype(str) + " (" + pct.astype(str) + "%)"

# 5. Enforce column order
table_all = table_all[["cluster_label", "total_count"] + existing_cols_all]

display(table_all)

In [ ]:
# 1. Filter the dataframe to keep ONLY the "No valid JSON reply" entries
invalid_json_df = narratives[narratives["narrative"] == "No valid JSON reply"].copy()

# 2. Drop the intermediate category column if it exists
invalid_json_df.drop(columns=["category"], inplace=True, errors='ignore')  

# 3. Export the dataframe to a new CSV file
invalid_json_df.to_csv("./out_files/05_tempi_df_R1_invalid_jsons.csv", index=False, encoding="utf-8-sig")

# Display the number of rows saved
print(f"Saved {len(invalid_json_df)}")

In [ ]:
invalids = pd.read_csv('./out_files/05_tempi_df_R1_invalid_jsons.csv', encoding='utf-8-sig')
invalids.info()

## RE Run Narrative Extraction for Problematic Rows

* Fixing bug in the function for the extraction

In [ ]:
import re

def extract_json_from_model_output(text: str):
    """
   Extracts the first valid JSON object from LLM output.
   """

    if not text or not isinstance(text, str):
        return None

    # --- 1. Remove common reasoning wrappers ---
    text = re.sub(r"<tool_call>.*?<tool_call>", "", text, flags=re.DOTALL)
    text = re.sub(r"```json", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text)

    # --- 2. Find all JSON-like blocks ---
    candidates = re.findall(r"\{[\s\S]*?\}", text)

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except Exception:
            continue

    return None


In [ ]:
import json
import pandas as pd
from botocore.config import Config
import time

import re

def extract_json_from_model_output(text: str):
    """
    Extracts the first valid JSON object from LLM output.
    """
    if not text or not isinstance(text, str):
        return None

    # --- 1. Remove common reasoning wrappers ---
    text = re.sub(r"<tool_call>.*?<tool_call>", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL) # <-- ADD THIS FOR DEEPSEEK R1
    text = re.sub(r"```json", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text)

    # --- 2. Find all JSON-like blocks ---
    candidates = re.findall(r"\{[\s\S]*?\}", text)

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except Exception:
            continue

    return None


# --- Load dataset ---
df = pd.read_csv("./out_files/05_tempi_df_R1_invalid_jsons.csv", encoding="utf-8-sig")

# Add output column
df["narrative"] = ""

# --- Main loop ---
for idx, row in df.iterrows():
    sitting_date = row.get("sitting_date", "")
    political_party = row.get("party", "")
    paragraph = row.get("paragraph", "")

    # --- Prompt construction ---
    prompt = f"""

Act like a senior political discourse analyst and annotation specialist with expertise in narrative extraction and framing analysis.

Your goal is to identify and distill a political narrative—defined as a recurring, specific overt or implicit claim that promotes a clear interpretation of an ongoing political issue and functions as both a cognitive and strategic tool—into a single, highly precise sentence.

Task: Read one Greek parliamentary speech and output its core narrative about the Tempi train crash (Greece, 2023) or return "irrelevant".

Inputs you receive (per row):
- sitting_date: {sitting_date}
- political_party: {political_party}
- paragraph: {paragraph}

Requirements:
1) Work exclusively from the speech; use other fields only for minimal contextual clarification.
2) A valid narrative must reflect a recurring or generalizable claim with a clear, specific interpretation (e.g., concrete accountability, causal mechanism, or policy implication), not vague or generic statements.
3) Avoid abstract or broad wording; prefer concrete, content-rich phrasing tied to identifiable issues (e.g., safety failures, institutional responsibility, regulatory gaps).
4) If the speech does not substantively address the Tempi crash or its direct implications (accountability, investigation, safety, governance, legal consequences), output "irrelevant".
5) Output a neutral, declarative statement expressing the core narrative without attribution or speech-reporting verbs.
6) Do not include names, actors, quotes, slogans, rhetorical questions, or multiple sentences.
7) English only. Plain text. Maximum 8 words.

Process (apply sequentially):
1) Assess whether the speech contains a narrative related to the Tempi crash and its systemic or political implications.
2) If absent, classify as "irrelevant".
3) If present, identify the underlying recurring claim and its specific interpretation (cause, responsibility, or solution).
4) Reformulate it into a concise, concrete proposition capturing its strategic or cognitive function.
5) Eliminate vagueness by replacing general terms with specific mechanisms or implications.

Constraints:
- Format: JSON only
- Style: minimal, declarative, analytical, specific
- Scope: include only the core narrative claim;
- Reasoning: think step-by-step internally, but output only final result
- Self-check: ensure one sentence, ≤8 words, highly specific, no attribution, valid JSON

Output format (strict):
{{
  "narrative": "your one-sentence result here or the exact word irrelevant"
}}

Take a deep breath and work on this problem step-by-step.

"""

# Send to DeepSeek R1 Instruct via Bedrock
    try:
      response = bedrock_runtime.converse(
          modelId="us.deepseek.r1-v1:0",
          messages=[
            {"role": "user", "content": [{"text": prompt}]}
        ]
      )

      model_reply = response["output"]["message"]["content"][0]["text"]

      parsed_data = extract_json_from_model_output(model_reply)
      
      if parsed_data is not None and "narrative" in parsed_data:
          result = parsed_data
      else:
          result = {"narrative": "No valid JSON reply"}
      # -----------------------------------

      df.at[idx, "narrative"] = result.get("narrative", "")
      print(f"Row {idx+1} annotated | {result.get('narrative', '')}")

    except Exception as e:
      print(f"Error on row {idx+1}: {e}")
      continue

    time.sleep(0.3)  # polite pacing

# --- Save results ---
output_file = "./out_files/05_tempi_df_R1_extra_narratives.csv"
df.to_csv(output_file, index=False)
print(f"\nAnnotation complete. Results saved to {output_file}")


## Merge new narratives

In [ ]:
extra_narratives_df = pd.read_csv("./out_files/05_tempi_df_R1_extra_narratives.csv", encoding='utf-8-sig')
extra_narratives_df.info()
extra_narratives_df.head(2)

In [ ]:
# 1. Create a dictionary from the new dataframe to map 'original_index' to the new 'narrative'
# This doesn't change the dataframe at all, just creates a lookup table
new_narratives_map = dict(zip(extra_narratives_df['original_index'], extra_narratives_df['narrative']))

# 2. Create a condition (mask) to find the exact rows to update in the main dataframe
# Condition 1: The narrative is currently "No valid JSON reply"
# Condition 2: The 'original_index' number exists in our new map
mask = (narratives['narrative'] == 'No valid JSON reply') & (narratives['original_index'].isin(new_narratives_map.keys()))

# 3. Apply the new narratives only to the rows that match the mask
narratives.loc[mask, 'narrative'] = narratives.loc[mask, 'original_index'].map(new_narratives_map)

# 4. Quick check to see the results
updated_count = mask.sum()
remaining_errors = (narratives['narrative'] == 'No valid JSON reply').sum()

print(f"Successfully updated {updated_count} rows.")
print(f"There are {remaining_errors} 'No valid JSON reply' entries remaining.")

In [ ]:
import pandas as pd

# 1. Classify all entries
narratives["category"] = narratives["narrative"].apply(
    lambda x: x if x in ["irrelevant", "No valid JSON reply"] else "actual_narrative"
)

# 2. Group, count, and pivot
table_all = (
    narratives.groupby(["cluster_label", "category"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# --- THE FIX STARTS HERE ---
# Define our columns early so we can use them for the sum
cols_all = ["actual_narrative", "irrelevant", "No valid JSON reply"]
existing_cols_all = [col for col in cols_all if col in table_all.columns]

# 3. Calculate total count per row (FIXED)
# We ONLY sum the specific category columns, explicitly ignoring cluster_label
table_all["total_count"] = table_all[existing_cols_all].sum(axis=1)
# --- THE FIX ENDS HERE ---


# 3.5 Calculate the overall sum and append as a new row
numeric_cols_to_sum = ["total_count"] + existing_cols_all

# Calculate the sum for the numeric columns
total_sums = table_all[numeric_cols_to_sum].sum()
total_sums["cluster_label"] = "Total" # Label the row

# Append the new row to the bottom of the dataframe
table_all.loc[len(table_all)] = total_sums
# ---------------------------------------------------------

# 4. Format cells to show "Count (Percentage%)"
for col in existing_cols_all:
    count = table_all[col]
    pct = (count / table_all["total_count"] * 100).round(2)
    # Combine count and percentage into a single string
    table_all[col] = count.astype(str) + " (" + pct.astype(str) + "%)"

# 5. Enforce column order
table_all = table_all[["cluster_label", "total_count"] + existing_cols_all]

display(table_all)

In [ ]:
# 1. Filter the dataframe to keep ONLY the actual narratives

actual_narratives_df = narratives[~narratives["narrative"].isin(["irrelevant", "No valid JSON reply"])].copy()

# 2. Export the clean dataframe to a new CSV file

actual_narratives_df.drop(columns=["category"], inplace=True)  # Drop the intermediate category column if not needed

actual_narratives_df.to_csv("./out_files/06_tempi_df_R1_final_narratives.csv", index=False, encoding="utf-8-sig")

# Display the number of rows saved
print(f"Saved {len(actual_narratives_df)}")

# Generate Narrative sample for Annotation

In [1]:
import pandas as pd
narratives = pd.read_csv('./out_files/06_tempi_df_R1_final_narratives.csv', encoding='utf-8-sig')
narratives.info()

<class 'pandas.DataFrame'>
RangeIndex: 4537 entries, 0 to 4536
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   original_index  4537 non-null   int64  
 1   member_name     4537 non-null   str    
 2   party           4537 non-null   str    
 3   sitting_date    4537 non-null   str    
 4   member_gender   4537 non-null   str    
 5   speech_id       4537 non-null   str    
 6   paragraph_id    4537 non-null   str    
 7   paragraph       4537 non-null   str    
 8   par_clean       4537 non-null   str    
 9   cluster_label   4537 non-null   int64  
 10  cake            4537 non-null   float64
 11  narrative       4537 non-null   str    
dtypes: float64(1), int64(2), str(9)
memory usage: 425.5 KB


In [7]:
import os 
import pandas as pd

os.makedirs('./out_files/annotation/', exist_ok=True)

# 1. Set your maximum character limit
MAX_LENGTH = 2000  # Change this to whatever your limit is!

# 2. FILTER FIRST: Create a pool of only the rows that fit your length requirement.
# We use .astype(str) to prevent errors if there are any empty cells.
valid_pool = narratives[narratives['paragraph'].astype(str).str.len() <= MAX_LENGTH]

# Optional: Check if you have enough valid rows left to sample 100
if len(valid_pool) < 100:
    print(f"Warning: Only {len(valid_pool)} valid rows left after filtering!")

# 3. Sample 100 rows from this newly filtered, safe pool
annotation = valid_pool.sample(100, random_state=42)

# 4. Format your final dataframe
annotation = annotation[['sitting_date', 'party', 'paragraph', 'narrative']]
annotation['comments'] = ''

# 5. Clean up the index and export to CSV
annotation = annotation.reset_index(drop=True)
annotation.to_csv('./out_files/annotation/01_narratives_annotation.csv', encoding='utf-8-sig')

annotation

,sitting_date,party,paragraph,narrative,comments
0,2025-02,Independents,"Ξεκινώντας, κρίνω αναγκαίο να αναφερθώ σε δύο ...",Government obstructs transparent Tempi crash i...,
1,2025-06,Nea Aristera,"Κυρίες και κύριοι συνάδελφοι, έχουν περάσει πε...",Government obstruction prevents Tempi crash ac...,
2,2025-07,Nea Dimokratia,"Η Νέα Δημοκρατία, η μεγάλη αυτή παράταξη είναι...",Minister's Tempi role unproven by judicial evi...,
3,2024-03,Plefsi Eleftherias,Ο κύριος Πρωθυπουργός οφείλει να έρθει εδώ. Αυ...,Government cover-up obscures Tempi crash accou...,
4,2025-04,Nea Dimokratia,"Όταν, λοιπόν, αυτή η ιστορία σιγά-σιγά αρχίζει...",Cover-up allegations scapegoat government to d...,
...,...,...,...,...,...
95,2025-06,Nea Dimokratia,"Έρχομαι στον απόλυτο πολιτικό κατήφορο, τον πι...",Tragedy exploited via baseless conspiracies ca...,
96,2023-09,PASOK-KINAL,"Και σας λέω, λοιπόν, εγώ ευθέως: Είστε επικίνδ...",Government's Tempi scandal accountability fail...,
97,2025-07,SYRIZA-PS,"Άρα, βάσει της δικής σας αρμοδιότητας, κύριε Υ...","Government deflects railway accountability, ne...",
98,2024-04,Plefsi Eleftherias,Και δεν είναι πάνω από τους ανθρώπους στη Θεσσ...,Government prioritizes other agendas over Thes...,


In [8]:
import random

# 3. Sample 100 rows from this newly filtered, safe pool
annotation = valid_pool.sample(100, random_state=42)

# --- NEW STEP: Add the random wrong narrative ---
# Get a list of all 100 narratives currently in your sample
pool_of_narratives = annotation['narrative'].tolist()

def pick_random_wrong_narrative(correct_narrative):
    # Filter out the correct narrative so we only have wrong options left
    wrong_options = [n for n in pool_of_narratives if n != correct_narrative]
    
    # Pick a random narrative from the remaining options
    return random.choice(wrong_options)

# Apply the function to create the new column
annotation['random_narrative'] = annotation['narrative'].apply(pick_random_wrong_narrative)
# ------------------------------------------------

# 4. Format your final dataframe
# Note: I added 'random_narrative' to your column selection list here
annotation = annotation[['sitting_date', 'party', 'paragraph', 'narrative', 'random_narrative']]
annotation['comments'] = ''

# 5. Clean up the index and export to CSV
annotation = annotation.reset_index(drop=True)
annotation.to_csv('./out_files/annotation/01_narratives_annotation.csv', encoding='utf-8-sig')

annotation

,sitting_date,party,paragraph,narrative,random_narrative,comments
0,2025-02,Independents,"Ξεκινώντας, κρίνω αναγκαίο να αναφερθώ σε δύο ...",Government obstructs transparent Tempi crash i...,Railway safety requires legislative and govern...,
1,2025-06,Nea Aristera,"Κυρίες και κύριοι συνάδελφοι, έχουν περάσει πε...",Government obstruction prevents Tempi crash ac...,Government prioritizes other agendas over Thes...,
2,2025-07,Nea Dimokratia,"Η Νέα Δημοκρατία, η μεγάλη αυτή παράταξη είναι...",Minister's Tempi role unproven by judicial evi...,Absence of European safety systems caused prev...,
3,2024-03,Plefsi Eleftherias,Ο κύριος Πρωθυπουργός οφείλει να έρθει εδώ. Αυ...,Government cover-up obscures Tempi crash accou...,Insufficient fire safety planning endangers pu...,
4,2025-04,Nea Dimokratia,"Όταν, λοιπόν, αυτή η ιστορία σιγά-σιγά αρχίζει...",Cover-up allegations scapegoat government to d...,"Post-crash measures neglect root causes, exace...",
...,...,...,...,...,...,...
95,2025-06,Nea Dimokratia,"Έρχομαι στον απόλυτο πολιτικό κατήφορο, τον πι...",Tragedy exploited via baseless conspiracies ca...,Government's Tempi scandal accountability fail...,
96,2023-09,PASOK-KINAL,"Και σας λέω, λοιπόν, εγώ ευθέως: Είστε επικίνδ...",Government's Tempi scandal accountability fail...,EU-mandated railway fragmentation caused syste...,
97,2025-07,SYRIZA-PS,"Άρα, βάσει της δικής σας αρμοδιότητας, κύριε Υ...","Government deflects railway accountability, ne...",Train tracking system implementation accelerat...,
98,2024-04,Plefsi Eleftherias,Και δεν είναι πάνω από τους ανθρώπους στη Θεσσ...,Government prioritizes other agendas over Thes...,Government obstruction prevents Tempi crash ac...,


## Create keys for annotation options

In [2]:
import pandas as pd

final = pd.read_csv('./out_files/annotation/narratives_clean_full.csv', encoding='utf-8-sig')

In [4]:
import pandas as pd
import random
import os

# Create directories
os.makedirs('./out_files/annotation/', exist_ok=True)
os.makedirs('./out_files/annotation/researcher_keys/', exist_ok=True)

def shuffle_row(row):
    # Group the three narratives with a label so we know which is which
    options = [
        ('target_model', row['narrative']),
        ('false_generated', row['false_narrative']),
        ('random_other', row['random_narrative'])
    ]
    
    # Shuffle the list randomly
    random.shuffle(options)
    
    # Find where the correct narrative ended up after shuffling (1, 2, or 3)
    correct_idx = [i for i, opt in enumerate(options) if opt[0] == 'target_model'][0]
    
    # Return the shuffled texts, the tracking keys, AND the static "None" option
    return pd.Series({
        'Option_1': options[0][1],
        'Option_2': options[1][1],
        'Option_3': options[2][1],
        'Option_4': 'None of the above',
        'Option_1_Type': options[0][0],
        'Option_2_Type': options[1][0],
        'Option_3_Type': options[2][0],
        'Option_4_Type': 'None of the above', 
        'Model_Option': f'Option_{correct_idx + 1}'
    })

# Apply the shuffling function to your dataset
# (Assuming your dataframe from the image is named 'final')
shuffled_data = final.apply(shuffle_row, axis=1)

# Merge the new columns back with the original context
df_processed = pd.concat([final[['sitting_date', 'party', 'paragraph']], shuffled_data], axis=1)

# Add the empty column where the annotator will make their dropdown selection
df_processed.insert(3, 'narrative', '')
df_processed['comments'] = ''

# ---------------------------------------------------------
# 1. EXPORT THE MASTER KEY (FOR YOU ONLY)
# ---------------------------------------------------------
df_processed.to_csv('./out_files/annotation/researcher_keys/master_key.csv', index=False, encoding='utf-8-sig')

# ---------------------------------------------------------
# 2. EXPORT THE BLINDED ANNOTATOR FILE
# ---------------------------------------------------------
# Included Option_4 in the final output
annotator_df = df_processed[['sitting_date', 'party', 'paragraph', 'narrative', 'Option_1', 'Option_2', 'Option_3', 'Option_4', 'comments']]
annotator_df.to_csv('./out_files/annotation/final_narratives_annotation_sheet.csv', index=False, encoding='utf-8-sig')

## Analysis of the annotation answers

In [5]:
import pandas as pd
import numpy as np
import os

# 1. Load the exact blinded sheet that matches your current master_key
blinded_df = pd.read_csv('./out_files/annotation/final_narratives_annotation_sheet.csv')

# 2. Simulate 3 annotators making selections
np.random.seed(42) # For reproducible random choices

for i in range(1, 4):
    annotator_mock = blinded_df.copy()
    
    # Randomly select Option 1, Option 2, Option 3, OR Option 4 ('None')
    choices = []
    for _, row in annotator_mock.iterrows():
        # Notice we added row['Option_4'] here!
        selected_text = np.random.choice([row['Option_1'], row['Option_2'], row['Option_3'], row['Option_4']])
        choices.append(selected_text)
        
    annotator_mock['narrative'] = choices
    
    # Save as a completed annotation file
    os.makedirs('./out_files/annotation/answers/', exist_ok=True)
    annotator_mock.to_csv(f'./out_files/annotation/answers/annotator_{i}.csv', index=False, encoding='utf-8-sig')

print("Created 3 fresh simulated annotator files with 'None' options included!")

Created 3 fresh simulated annotator files with 'None' options included!


In [6]:
import pandas as pd
import glob
import os

# 1. Load your Master Key
master_key = pd.read_csv('./out_files/annotation/researcher_keys/master_key.csv')

# Drop the empty overlapping columns from the master key so they don't clash with the annotator's file
master_key = master_key.drop(columns=['narrative', 'comments'], errors='ignore')

def decode_selection(row):
    choice = row['narrative']
    
    if choice == row['Option_1']:
        return row['Option_1_Type']
    elif choice == row['Option_2']:
        return row['Option_2_Type']
    elif choice == row['Option_3']:
        return row['Option_3_Type']
    elif choice == row['Option_4']:         
        return row['Option_4_Type']
    else:
        return 'Invalid/Blank'

# 2. Find all completed annotator files
annotator_files = glob.glob('./out_files/annotation/answers/annotator_*.csv')

# 3. Process each file
for file in annotator_files:
    annotator_name = os.path.basename(file).replace('.csv', '')
    
    annotator_df = pd.read_csv(file)
    
    # Merge the annotator's choices with the cleaned master key
    merged_df = pd.merge(
        master_key, 
        annotator_df[['paragraph', 'narrative', 'comments']], 
        on='paragraph', 
        how='left'
    )
    
    # 4. UN-SHUFFLE: Apply the decoding function
    merged_df['selection_type'] = merged_df.apply(decode_selection, axis=1)
    
    # Updated to count 'target_model' based on your new labels
    correct_count = (merged_df['selection_type'] == 'target_model').sum()
    total_count = len(merged_df)
    accuracy = (correct_count / total_count) * 100
    
    print(f"--- Results for {annotator_name} ---")
    print(f"Accuracy: {accuracy:.1f}% ({correct_count}/{total_count} the model was chosen)")
    print(merged_df['selection_type'].value_counts())
    print("-" * 30 + "\n")
    
    # 5. Save the unshuffled results
    final_analysis_df = merged_df[['sitting_date', 'party', 'paragraph', 'narrative', 'selection_type', 'comments']]
    
    os.makedirs('./out_files/annotation/analysis/', exist_ok=True)
    final_analysis_df.to_csv(f'./out_files/annotation/analysis/{annotator_name}_evaluated.csv', index=False, encoding='utf-8-sig')

print("All files unshuffled and saved to ./out_files/annotation/analysis/")

--- Results for annotator_1 ---
Accuracy: 22.0% (22/100 the model was chosen)
selection_type
random_other         32
None of the above    30
target_model         22
false_generated      16
Name: count, dtype: int64
------------------------------

--- Results for annotator_2 ---
Accuracy: 28.0% (28/100 the model was chosen)
selection_type
target_model         28
random_other         27
None of the above    24
false_generated      21
Name: count, dtype: int64
------------------------------

--- Results for annotator_3 ---
Accuracy: 24.0% (24/100 the model was chosen)
selection_type
random_other         30
None of the above    27
target_model         24
false_generated      19
Name: count, dtype: int64
------------------------------

All files unshuffled and saved to ./out_files/annotation/analysis/


In [39]:
import pandas as pd
import glob
import os
import itertools
from sklearn.metrics import cohen_kappa_score

# 1. Find all evaluated annotator files
evaluated_files = sorted(glob.glob('./out_files/annotation/analysis/annotator_*.csv'))

# 2. Load the data and extract just the selection types, using 'paragraph' as the anchor
annotator_data = {}
for file in evaluated_files:
    annotator_name = os.path.basename(file).replace('.csv', '')
    df = pd.read_csv(file)
    
    # We set 'paragraph' as the index so we can perfectly align the rows across all files
    df = df.set_index('paragraph')
    annotator_data[annotator_name] = df['selection_type']

# 3. Combine everything into a single DataFrame for easy comparison
combined_df = pd.DataFrame(annotator_data)

# Drop any rows where an annotator might have missed a question (just to be safe)
combined_df = combined_df.dropna()

# 4. Generate all possible pairs of annotators (e.g., 1 vs 2, 1 vs 3, 2 vs 3)
annotators = list(combined_df.columns)
pairs = list(itertools.combinations(annotators, 2))

# 5. Calculate and print Pairwise Cohen's Kappa
print("--- Pairwise Cohen's Kappa Scores ---")
kappa_scores = []

for a1, a2 in pairs:
    score = cohen_kappa_score(combined_df[a1], combined_df[a2])
    kappa_scores.append(score)
    print(f"{a1} vs {a2}: {score:.4f}")

print("-" * 37)

# Calculate the average Kappa across all pairs
average_kappa = sum(kappa_scores) / len(kappa_scores)
print(f"Average Cohen's Kappa: {average_kappa:.4f}")

--- Pairwise Cohen's Kappa Scores ---
annotator_1_evaluated vs annotator_2_evaluated: -0.0120
annotator_1_evaluated vs annotator_3_evaluated: 0.0365
annotator_2_evaluated vs annotator_3_evaluated: -0.0238
-------------------------------------
Average Cohen's Kappa: 0.0002
